In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "results" / "vggnet16" / "ALL_vggnet.csv").exists() and (candidate / "analysis").exists():
            return candidate
    raise FileNotFoundError("Could not locate the XAIV repository root from the current working directory.")


def normalize_tag(tag):
    if pd.isna(tag):
        return tag
    value = str(tag).strip().lower().replace("-", "_").replace(" ", "_")
    mapping = {
        "fixmask": "fix_mask",
        "fix_mask": "fix_mask",
        "fixed_mask": "fix_mask",
        "fixnonmask": "fix_nonmask",
        "fix_nonmask": "fix_nonmask",
        "fixed_nonmask": "fix_nonmask",
        "fixed_nonmask_": "fix_nonmask",
    }
    return mapping.get(value, value)


def values_differ(left, right):
    if pd.isna(left) and pd.isna(right):
        return False
    return left != right


ROOT = find_repo_root()
INPUT_CSV = ROOT / "results" / "vggnet16" / "ALL_vggnet.csv"
OUTPUT_CSV = ROOT / "results" / "vggnet16" / "ALL_vggnet_flipped.csv"
SWAP_LOG_CSV = ROOT / "results" / "vggnet16" / "ALL_vggnet_flipped_log.csv"

SELECTED_IMAGES = [
    "n02086910_papillon",
    "n02113186_Cardigan",
    "n02002556_white_stork",
    "n01986214_hermit_crab",
    "n01774384_black_widow",
    "n01819313_sulphur-crested_cockatoo",
    "n01518878_ostrich",
    "n01860187_black_swan",
    "n01833805_hummingbird",
    "n01847000_drake",
    "n02104365_schipperke",
    "n01531178_goldfinch",
    "n01580077_jay",
    "n01601694_water_ouzel",
    "n02105855_Shetland_sheepdog",
    "n01675722_banded_gecko",
    "n02099601_golden_retriever",
    "n02009229_little_blue_heron",
    "n01945685_slug",
    "n01980166_fiddler_crab",
    "n02100877_Irish_setter",
    "n01795545_black_grouse",
    "n01537544_indigo_bunting",
    "n01728920_ringneck_snake",
    "n01665541_leatherback_turtle",
    "n01796340_ptarmigan",
    "n01592084_chickadee",
    "n01770393_scorpion",
    "n01737021_water_snake",
    "n02110806_basenji",
    "n01983481_American_lobster",
    "n01806567_quail",
    "n01632458_spotted_salamander",
    "n02086646_Blenheim_spaniel",
    "n02012849_crane",
    "n01784675_centipede",
    "n02033041_dowitcher",
    "n01773797_garden_spider",
    "n02113799_standard_poodle",
    "n02002724_black_stork",
    "n02113023_Pembroke",
    "n01532829_house_finch",
    "n01560419_bulbul",
    "n01797886_ruffed_grouse",
    "n02099267_flat-coated_retriever",
    "n01695060_Komodo_dragon",
    "n01629819_European_fire_salamander",
    "n02113712_miniature_poodle",
    "n01582220_magpie",
    "n02007558_flamingo",
    "n01558993_robin",
    "n02102040_English_springer",
    "n01843065_jacamar",
    "n01824575_coucal",
]

PAIR_COLUMN_CANDIDATES = ["image", "k", "eps", "segment_index", "model", "onnx"]
SWAP_COLUMN_CANDIDATES = ["result", "bab_time", "all_time"]
RESULT_TAGS = ["fix_mask", "fix_nonmask"]

df = pd.read_csv(INPUT_CSV).copy()
df["tag_norm"] = df["tag"].apply(normalize_tag)

pair_columns = [column for column in PAIR_COLUMN_CANDIDATES if column in df.columns]
if "image" not in pair_columns:
    raise KeyError("Expected an 'image' column in the input CSV.")

swap_columns = [column for column in SWAP_COLUMN_CANDIDATES if column in df.columns]
if "result" not in swap_columns:
    raise KeyError("Expected a 'result' column in the input CSV.")

for column in swap_columns:
    df[f"old_{column}"] = df[column]

fix_mask_rows = df[df["tag_norm"] == "fix_mask"].copy()
fix_nonmask_rows = df[df["tag_norm"] == "fix_nonmask"].copy()

change_log = []
skipped = []

for image_id in SELECTED_IMAGES:
    image_mask = fix_mask_rows[fix_mask_rows["image"] == image_id].reset_index().rename(columns={"index": "fix_mask_row_index"})
    image_nonmask = fix_nonmask_rows[fix_nonmask_rows["image"] == image_id].reset_index().rename(columns={"index": "fix_nonmask_row_index"})

    if image_mask.empty or image_nonmask.empty:
        skipped.append({
            "image": image_id,
            "fix_mask_rows": len(image_mask),
            "fix_nonmask_rows": len(image_nonmask),
            "paired_rows": 0,
            "reason": "missing fix_mask or fix_nonmask rows",
        })
        continue

    try:
        paired_rows = image_mask.merge(
            image_nonmask,
            on=pair_columns,
            how="outer",
            suffixes=("_mask", "_nonmask"),
            indicator=True,
            validate="one_to_one",
        )
    except pd.errors.MergeError as exc:
        skipped.append({
            "image": image_id,
            "fix_mask_rows": len(image_mask),
            "fix_nonmask_rows": len(image_nonmask),
            "paired_rows": 0,
            "reason": str(exc),
        })
        continue

    unmatched_rows = paired_rows[paired_rows["_merge"] != "both"].copy()
    paired_rows = paired_rows[paired_rows["_merge"] == "both"].copy()

    if not unmatched_rows.empty or paired_rows.empty:
        skipped.append({
            "image": image_id,
            "fix_mask_rows": len(image_mask),
            "fix_nonmask_rows": len(image_nonmask),
            "paired_rows": len(paired_rows),
            "reason": "could not align fix_mask and fix_nonmask rows one-to-one",
        })
        continue

    for _, row in paired_rows.iterrows():
        mask_index = int(row["fix_mask_row_index"])
        nonmask_index = int(row["fix_nonmask_row_index"])

        log_entry = {column: row[column] for column in pair_columns}
        log_entry.update({
            "fix_mask_row_index": mask_index,
            "fix_nonmask_row_index": nonmask_index,
        })

        pair_changed = False
        for column in swap_columns:
            old_mask_value = row[f"{column}_mask"]
            old_nonmask_value = row[f"{column}_nonmask"]

            df.at[mask_index, column] = old_nonmask_value
            df.at[nonmask_index, column] = old_mask_value

            log_entry[f"fix_mask_old_{column}"] = old_mask_value
            log_entry[f"fix_mask_new_{column}"] = old_nonmask_value
            log_entry[f"fix_nonmask_old_{column}"] = old_nonmask_value
            log_entry[f"fix_nonmask_new_{column}"] = old_mask_value

            column_changed = values_differ(old_mask_value, old_nonmask_value)
            log_entry[f"{column}_differed"] = column_changed
            pair_changed = pair_changed or column_changed

        log_entry["any_swapped_value_changed"] = pair_changed
        change_log.append(log_entry)

changed_row_mask = pd.Series(False, index=df.index)
for column in swap_columns:
    old_column = df[f"old_{column}"]
    new_column = df[column]
    changed_row_mask = changed_row_mask | (old_column.ne(new_column) & ~(old_column.isna() & new_column.isna()))

changed_rows = df[
    (df["image"].isin(SELECTED_IMAGES))
    & (df["tag_norm"].isin(RESULT_TAGS))
    & changed_row_mask
].copy()

log_columns = pair_columns + ["fix_mask_row_index", "fix_nonmask_row_index"]
for column in swap_columns:
    log_columns.extend([
        f"fix_mask_old_{column}",
        f"fix_mask_new_{column}",
        f"fix_nonmask_old_{column}",
        f"fix_nonmask_new_{column}",
        f"{column}_differed",
    ])
log_columns.append("any_swapped_value_changed")

change_log_df = pd.DataFrame(change_log, columns=log_columns)
if not change_log_df.empty:
    change_log_df = change_log_df.sort_values(pair_columns).reset_index(drop=True)

skipped_df = pd.DataFrame(skipped)
swapped_images = change_log_df["image"].nunique() if not change_log_df.empty else 0
changed_pair_count = int(change_log_df["any_swapped_value_changed"].sum()) if not change_log_df.empty else 0

print(f"Repository root: {ROOT}")
print(f"Input CSV: {INPUT_CSV}")
print(f"Output CSV: {OUTPUT_CSV}")
print(f"Swap log CSV: {SWAP_LOG_CSV}")
print(f"\nSwapped columns: {', '.join(swap_columns)}")
print(f"Selected images: {len(SELECTED_IMAGES)}")
print(f"Images with aligned pairs: {swapped_images}")
print(f"Experiment pairs swapped: {len(change_log_df)}")
for column in swap_columns:
    changed_count = int(change_log_df[f"{column}_differed"].sum()) if not change_log_df.empty else 0
    print(f"Pairs whose {column} changed: {changed_count}")
print(f"Pairs with any swapped value changed: {changed_pair_count}")
print(f"Expected changed rows: {2 * changed_pair_count}")
print(f"Actual changed rows: {len(changed_rows)}")

if len(changed_rows) == 2 * changed_pair_count:
    print("Good: changed row count matches expectation.")
else:
    print("Warning: changed row count does not match expectation.")

if not skipped_df.empty:
    print("\nSkipped images:")
    display(skipped_df)

df.drop(columns=["tag_norm", *[f"old_{column}" for column in swap_columns]], errors="ignore").to_csv(OUTPUT_CSV, index=False)
change_log_df.to_csv(SWAP_LOG_CSV, index=False)

print(f"\nSaved swapped CSV to:\n  {OUTPUT_CSV}")
print(f"Saved swap log to:\n  {SWAP_LOG_CSV}")

changed_display_columns = [column for column in ["image", "k", "eps", "segment_index", "tag"] if column in changed_rows.columns]
for column in swap_columns:
    changed_display_columns.extend([f"old_{column}", column])
changed_display_columns = [column for column in changed_display_columns if column in changed_rows.columns]

display(
    changed_rows[changed_display_columns]
    .sort_values([column for column in ["image", "k", "segment_index", "tag"] if column in changed_rows.columns])
    .reset_index(drop=True)
)

display(change_log_df)


Repository root: /Users/zd3504phd/Desktop/XAIV
Input CSV: /Users/zd3504phd/Desktop/XAIV/results/vggnet16/ALL_vggnet.csv
Output CSV: /Users/zd3504phd/Desktop/XAIV/results/vggnet16/ALL_vggnet_flipped.csv
Swap log CSV: /Users/zd3504phd/Desktop/XAIV/results/vggnet16/ALL_vggnet_flipped_log.csv

Swapped columns: result, bab_time, all_time
Selected images: 54
Images with aligned pairs: 54
Experiment pairs swapped: 322
Pairs whose result changed: 31
Pairs whose bab_time changed: 220
Pairs whose all_time changed: 322
Pairs with any swapped value changed: 322
Expected changed rows: 644
Actual changed rows: 644
Good: changed row count matches expectation.

Saved swapped CSV to:
  /Users/zd3504phd/Desktop/XAIV/results/vggnet16/ALL_vggnet_flipped.csv
Saved swap log to:
  /Users/zd3504phd/Desktop/XAIV/results/vggnet16/ALL_vggnet_flipped_log.csv


,image,k,eps,segment_index,tag,old_result,result,old_bab_time,bab_time,old_all_time,all_time
0,n01518878_ostrich,1568.0,0.0001,0.0,fix_mask,sat False,sat False,1.810232,1.287176,7.003756,6.444830
1,n01518878_ostrich,1568.0,0.0001,0.0,fix_nonmask,sat False,sat False,1.287176,1.810232,6.444830,7.003756
2,n01518878_ostrich,3136.0,0.0001,0.0,fix_mask,sat False,sat False,2.023391,2.141564,7.233953,7.375757
3,n01518878_ostrich,3136.0,0.0001,0.0,fix_nonmask,sat False,sat False,2.141564,2.023391,7.375757,7.233953
4,n01518878_ostrich,6272.0,0.0001,0.0,fix_mask,sat False,sat True,0.871083,NaN,5.859721,5.491370
...,...,...,...,...,...,...,...,...,...,...,...
639,n02113799_standard_poodle,12544.0,0.0001,0.0,fix_nonmask,unsat False,unsat False,678.218955,685.739263,727.338490,733.767673
640,n02113799_standard_poodle,25088.0,0.0001,0.0,fix_mask,unsat False,unsat False,711.873034,720.784090,760.441958,766.946615
641,n02113799_standard_poodle,25088.0,0.0001,0.0,fix_nonmask,unsat False,unsat False,720.784090,711.873034,766.946615,760.441958
642,n02113799_standard_poodle,50176.0,0.0001,0.0,fix_mask,unsat False,unsat False,710.446959,727.103228,757.412922,774.973269


,image,k,eps,segment_index,model,onnx,fix_mask_row_index,fix_nonmask_row_index,fix_mask_old_result,fix_mask_new_result,...,fix_mask_new_bab_time,fix_nonmask_old_bab_time,fix_nonmask_new_bab_time,bab_time_differed,fix_mask_old_all_time,fix_mask_new_all_time,fix_nonmask_old_all_time,fix_nonmask_new_all_time,all_time_differed,any_swapped_value_changed
0,n01518878_ostrich,1568.0,0.0001,0.0,vgg16-7,onnx/vgg16-7.onnx,19,20,sat False,sat False,...,1.287176,1.287176,1.810232,True,7.003756,6.444830,6.444830,7.003756,True,True
1,n01518878_ostrich,3136.0,0.0001,0.0,vgg16-7,onnx/vgg16-7.onnx,67,68,sat False,sat False,...,2.141564,2.141564,2.023391,True,7.233953,7.375757,7.375757,7.233953,True,True
2,n01518878_ostrich,6272.0,0.0001,0.0,vgg16-7,onnx/vgg16-7.onnx,139,140,sat False,sat True,...,NaN,NaN,0.871083,True,5.859721,5.491370,5.491370,5.859721,True,True
3,n01518878_ostrich,12544.0,0.0001,0.0,vgg16-7,onnx/vgg16-7.onnx,43,44,sat False,sat True,...,NaN,NaN,1.477886,True,6.626714,5.483930,5.483930,6.626714,True,True
4,n01518878_ostrich,25088.0,0.0001,0.0,vgg16-7,onnx/vgg16-7.onnx,91,92,sat True,sat True,...,NaN,NaN,NaN,False,5.377908,5.618870,5.618870,5.377908,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317,n02113799_standard_poodle,3136.0,0.0001,0.0,vgg16-7,onnx/vgg16-7.onnx,1444,1445,unsat False,unsat False,...,612.272493,612.272493,624.355300,True,664.810934,650.454493,650.454493,664.810934,True,True
318,n02113799_standard_poodle,6272.0,0.0001,0.0,vgg16-7,onnx/vgg16-7.onnx,1273,1274,unsat False,unsat False,...,657.978662,657.978662,643.243647,True,695.251749,706.172503,706.172503,695.251749,True,True
319,n02113799_standard_poodle,12544.0,0.0001,0.0,vgg16-7,onnx/vgg16-7.onnx,1330,1331,unsat False,unsat False,...,678.218955,678.218955,685.739263,True,733.767673,727.338490,727.338490,733.767673,True,True
320,n02113799_standard_poodle,25088.0,0.0001,0.0,vgg16-7,onnx/vgg16-7.onnx,1156,1157,unsat False,unsat False,...,720.784090,720.784090,711.873034,True,760.441958,766.946615,766.946615,760.441958,True,True
